## Interval Analysis

In [1]:
# !pip install tensorboardX

import os

import certifi
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms

os.environ["SSL_CERT_FILE"] = certifi.where()
# from tensorboardX import SummaryWriter

use_cuda = True
device = torch.device("xpu" if use_cuda else "cpu")
batch_size = 64

np.random.seed(42)
torch.manual_seed(42)


## Dataloaders
train_dataset = datasets.MNIST(
    "mnist_data/",
    train=True,
    download=True,
    transform=transforms.Compose([transforms.ToTensor()]),
)
test_dataset = datasets.MNIST(
    "mnist_data/",
    train=False,
    download=True,
    transform=transforms.Compose([transforms.ToTensor()]),
)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False
)


## Simple NN. You can change this if you want. If you change it, mention the architectural details in your report.
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc = nn.Linear(28 * 28, 200)
        self.fc2 = nn.Linear(200, 10)

    def forward(self, x):
        x = x.view((-1, 28 * 28))
        x = F.relu(self.fc(x))
        x = self.fc2(x)
        x = F.softmax(x, dim=-1)  # added softmax for probabilities
        return x


class Normalize(nn.Module):
    def forward(self, x):
        return (x - 0.1307) / 0.3081


# Add the data normalization as a first "layer" to the network
# this allows us to search for adverserial examples to the real image, rather than
# to the normalized image
model = nn.Sequential(Normalize(), Net())

model = model.to(device)
model.train()


Sequential(
  (0): Normalize()
  (1): Net(
    (fc): Linear(in_features=784, out_features=200, bias=True)
    (fc2): Linear(in_features=200, out_features=10, bias=True)
  )
)

In [2]:
def train_model(model, num_epochs):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for i, data in enumerate(train_loader, 0):
            images, labels = data
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(
            f"Epoch {epoch + 1}/{num_epochs}, Loss: {running_loss / len(train_loader):.3f}"
        )


def test_model(model):
    model.eval()
    with torch.no_grad():
        correct = 0
        total = 0
        for data in test_loader:
            images, labels = data
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        print(f"Accuracy on images: {100 * correct / total}")

In [3]:
train_model(model, 15)
test_model(model)

Epoch 1/15, Loss: 2.012
Epoch 2/15, Loss: 1.729
Epoch 3/15, Loss: 1.627
Epoch 4/15, Loss: 1.596
Epoch 5/15, Loss: 1.582
Epoch 6/15, Loss: 1.572
Epoch 7/15, Loss: 1.566
Epoch 8/15, Loss: 1.561
Epoch 9/15, Loss: 1.557
Epoch 10/15, Loss: 1.553
Epoch 11/15, Loss: 1.550
Epoch 12/15, Loss: 1.547
Epoch 13/15, Loss: 1.545
Epoch 14/15, Loss: 1.542
Epoch 15/15, Loss: 1.540
Accuracy on images: 93.43


### Write the interval analysis for the simple model

In [4]:
def box(x: torch.Tensor, epsilon: float) -> tuple[torch.Tensor, torch.Tensor]:
    return (x - epsilon).clamp(0, 1), (x + epsilon).clamp(0, 1)


# Adapted from https://github.com/Zinoex/bound_propagation/blob/main/src/bound_propagation/linear.py
def prop_layer(
    weights: torch.Tensor, bias: torch.Tensor, bounds: tuple[torch.Tensor, torch.Tensor]
) -> tuple[torch.Tensor, torch.Tensor]:
    bounds = tuple(x.flatten().to(device) for x in bounds)

    center = (bounds[0] + bounds[1]) / 2
    diff = (bounds[1] - bounds[0]) / 2

    with torch.no_grad():
        w_mid = center @ weights.T + bias
        w_diff = diff @ weights.T.abs()

    return w_mid - w_diff, w_mid + w_diff


def prop_relu(
    bounds: tuple[torch.Tensor, torch.Tensor],
) -> tuple[torch.Tensor, torch.Tensor]:
    return (F.relu(bounds[0]), F.relu(bounds[1]))


def check(x: torch.Tensor, y: torch.Tensor, epsilon: float) -> bool:
    bounds = box(x, epsilon)

    bounds_norm = tuple(model[0](x) for x in bounds)

    bounds_layer1 = prop_layer(model[1].fc.weight, model[1].fc.bias, bounds_norm)
    bounds_relu = prop_relu(bounds_layer1)
    bounds_layer2 = prop_layer(model[1].fc2.weight, model[1].fc2.bias, bounds_relu)

    # As a matrix, subtract by each label out from the correct out
    out_layer = torch.diag(-torch.ones(10)).to(device)
    out_layer[:, y] += 1

    bounds_output = prop_layer(out_layer, torch.zeros(10).to(device), bounds_layer2)

    return bool((bounds_output[0] >= 0).all())

In [ ]:
for eps in torch.linspace(0.01, 0.1, steps=10):
    correct = 0
    total = 0
    for images, labels in test_loader:
        for image, label in zip(images, labels):
            if check(image[0], label, eps.item()):
                correct += 1
            total += 1

    print(f"Correct for {eps:.4}: {correct / total}")

Correct for 0.01: 0.4959
Correct for 0.02: 0.053
Correct for 0.03: 0.002
Correct for 0.04: 0.0
Correct for 0.05: 0.0
Correct for 0.06: 0.0
Correct for 0.07: 0.0
Correct for 0.08: 0.0
Correct for 0.09: 0.0
Correct for 0.1: 0.0
